# 03 — E1: Crash-Free Baseline

**Experiment 1**: All 100 nodes stay alive for 1000 cycles.
- HEAL vs Ring vs Gossip convergence speed
- Validates HEAL achieves FL-grade convergence even without central server
- Expected: HEAL ≈ 75–80% top-1 acc, Gossip ≈ 65%, Ring ≈ 55%

In [ ]:
import subprocess, sys, os

# ── Resolve repo root (works locally and on Colab) ────────────────────────
if os.path.exists("/content"):
    os.chdir("/content")
    subprocess.check_call(["rm", "-rf", "/content/fl-rockstar"])
    subprocess.check_call(["git", "clone",
        "https://github.com/51103220/fl-rockstar.git",
        "/content/fl-rockstar"])
    os.chdir("/content/fl-rockstar")

    # Mount Google Drive for persistent checkpoints
    from google.colab import drive
    drive.mount('/content/drive')
    CKPT_DIR = '/content/drive/MyDrive/fl_checkpoints/e1'
else:
    repo_root = os.getcwd()
    while repo_root != "/" and not os.path.isdir(os.path.join(repo_root, "src")):
        repo_root = os.path.dirname(repo_root)
    os.chdir(repo_root)
    CKPT_DIR = 'checkpoints/e1'

os.makedirs(CKPT_DIR, exist_ok=True)

print("=" * 60)
repo_root = os.getcwd()
print(f"[1] cwd          : {repo_root}")
print(f"[2] python       : {sys.executable}")
print(f"[3] sys.version  : {sys.version.split()[0]}")
print(f"[4] src/ exists  : {os.path.isdir(os.path.join(repo_root, 'src'))}")
print(f"[5] ckpt_dir     : {CKPT_DIR}")

src_path = os.path.join(repo_root, "src")
assert os.path.isdir(src_path), f"src/ not found under {repo_root}"

if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

subprocess.check_call([sys.executable, "-m", "pip", "install",
    "torch", "torchvision", "datasets", "networkx",
    "numpy", "scipy", "matplotlib", "seaborn", "pandas",
    "tqdm", "pyyaml", "--quiet"])
print("[6] pip install  : OK")

import copy, random, json
import torch
import matplotlib.pyplot as plt
import numpy as np
print(f"[7] torch        : {torch.__version__}")
print(f"[8] cuda avail   : {torch.cuda.is_available()}")

from src.data import make_noniid_cifar10
from src.model import SimpleCNN
from src.topology import build_elevator_topology, build_ring_topology, build_gossip_topology
from src.simulation import HealClient, HealHub, run_simulation
print("[9] src imports  : OK")

SEED       = 42
N_NODES    = 20
N_HUBS     = 3
N_CYCLES   = 50
EVAL_EVERY = 5

device = "cuda" if torch.cuda.is_available() else "cpu"
random.seed(SEED)
torch.manual_seed(SEED)
print(f"[10] device      : {device}")
print(f"[11] config      : {N_NODES} nodes / {N_CYCLES} cycles / eval every {EVAL_EVERY}")
print("=" * 60)

In [2]:
node_data, test_loader = make_noniid_cifar10(N_NODES, shards_per_node=4, seed=SEED)  # 4 shards = less non-IID, more stable
init_model = SimpleCNN()
print(f'Model parameters: {sum(p.numel() for p in init_model.parameters()):,}')

Loading CIFAR-10 from HuggingFace (uoft-cs/cifar10)...


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


Model parameters: 620,810


In [ ]:
print('Running HEAL...')
G_heal, hub_ids = build_elevator_topology(N_NODES, N_HUBS, SEED)
hubs    = [HealHub(i,    copy.deepcopy(init_model), node_data[i], device=device) for i in hub_ids]
clients = [HealClient(i, copy.deepcopy(init_model), node_data[i], device=device)
           for i in range(N_NODES) if i not in hub_ids]
hist_heal = run_simulation('heal', N_CYCLES, clients, hubs, hub_ids,
                            G_heal, test_loader, eval_every=EVAL_EVERY,
                            n_sends=2, ckpt_dir=CKPT_DIR)
print(f'HEAL final acc: {hist_heal[-1][1]:.4f}')

In [ ]:
print('Running Ring...')
G_ring, _ = build_ring_topology(N_NODES)
clients_ring = [HealClient(i, copy.deepcopy(init_model), node_data[i], device=device) for i in range(N_NODES)]
hist_ring = run_simulation('ring', N_CYCLES, clients_ring, [], [],
                            G_ring, test_loader, eval_every=EVAL_EVERY,
                            ckpt_dir=CKPT_DIR)
print(f'Ring final acc: {hist_ring[-1][1]:.4f}')

In [ ]:
print('Running Gossip...')
G_gossip, _ = build_gossip_topology(N_NODES)
clients_gossip = [HealClient(i, copy.deepcopy(init_model), node_data[i], device=device) for i in range(N_NODES)]
hist_gossip = run_simulation('gossip', N_CYCLES, clients_gossip, [], [],
                              G_gossip, test_loader, eval_every=EVAL_EVERY,
                              ckpt_dir=CKPT_DIR)
print(f'Gossip final acc: {hist_gossip[-1][1]:.4f}')

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

for hist, label, color, ls in [
    (hist_heal,   'HEAL (Elevator)',  '#0284C7', '-'),
    (hist_gossip, 'Gossip',           '#F59E0B', '--'),
    (hist_ring,   'Ring (no healing)','#EF4444', ':'),
]:
    cycles, accs = zip(*hist)
    ax.plot(cycles, accs, label=label, color=color, linestyle=ls, linewidth=2)

ax.set_xlabel('Training Cycle', fontsize=12)
ax.set_ylabel('Average Test Accuracy', fontsize=12)
ax.set_title('E1: Crash-Free — HEAL vs Gossip vs Ring (CIFAR-10, Non-IID)', fontsize=14)
ax.legend(fontsize=11)
ax.grid(alpha=0.3)
ax.set_ylim(0, 1)
plt.tight_layout()
plt.savefig('../results/03_e1_crash_free.png', dpi=150)
plt.show()

print(f"\nSummary (cycle {N_CYCLES}):")
print(f"  HEAL:   {hist_heal[-1][1]:.4f}")
print(f"  Gossip: {hist_gossip[-1][1]:.4f}")
print(f"  Ring:   {hist_ring[-1][1]:.4f}")